# Voicebox + Qwen3-TTS for EPUB Player (free Colab trial)

This notebook runs the **actual open-source [jamiepine/voicebox](https://github.com/jamiepine/voicebox)** backend on a Google Colab GPU and gives the iPhone EPUB reader a temporary HTTPS connection.

**Before Run all:** Runtime → Change runtime type → choose a GPU (T4 is fine when Colab offers one).

Voicebox runs in an isolated **Python 3.11** environment, matching Voicebox's Docker image. Colab's own newer Python is used only to launch the notebook.

Keep this runtime alive while you listen. When Colab disconnects, run the notebook again and tap the new pairing button. Already-generated audiobook chunks remain cached in EPUB Player.


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. In Colab choose Runtime → Change runtime type → GPU, then Run all again.")


In [ ]:
# Install Voicebox in an isolated Python 3.11 environment.
import pathlib, subprocess, shutil, sys, sysconfig

ROOT = pathlib.Path("/content/voicebox")
VENV = pathlib.Path("/content/voicebox-py311")
PIN = "51f49dea198384b4eb6087b72c17057c6eb1c1cd"
MARKER = pathlib.Path("/content/.voicebox_epub_py311_v3_ready")

def run(cmd, cwd=None, env=None):
    cmd = list(map(str, cmd))
    print("+", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

# Always restore a clean Voicebox checkout; this removes any edits made
# by notebook assistants or previous troubleshooting attempts.
if not ROOT.exists():
    run(["git", "clone", "https://github.com/jamiepine/voicebox.git", str(ROOT)])
run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", PIN])
run(["git", "-C", str(ROOT), "checkout", "--detach", PIN])
run(["git", "-C", str(ROOT), "reset", "--hard", PIN])
run(["git", "-C", str(ROOT), "clean", "-fd"])

# Install uv with Colab's existing Python, then discover the executable
# instead of assuming a hard-coded /root/.local/bin location.
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"])
candidates = [
    shutil.which("uv"),
    str(pathlib.Path(sysconfig.get_path("scripts")) / "uv"),
    "/usr/local/bin/uv",
    "/root/.local/bin/uv",
]
UV = next((pathlib.Path(p) for p in candidates if p and pathlib.Path(p).is_file()), None)
if UV is None:
    raise RuntimeError("uv installed but its executable could not be located. Candidates checked: " + ", ".join(str(p) for p in candidates if p))
print("uv:", UV)

run([str(UV), "python", "install", "3.11"])

# If this is a retry after a failed/partial install, rebuild the venv.
if not MARKER.exists() and VENV.exists():
    shutil.rmtree(VENV)

if not VENV.exists():
    run([str(UV), "venv", "--python", "3.11", "--seed", str(VENV)])

PY = str(VENV / "bin" / "python")
print("Voicebox Python:", subprocess.check_output([PY, "--version"], text=True).strip())

if not MARKER.exists():
    run([PY, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    # Match Voicebox's own Docker install order.
    run([PY, "-m", "pip", "install", "-r", str(ROOT / "backend" / "requirements.txt")])
    run([PY, "-m", "pip", "install", "--no-deps", "chatterbox-tts"])
    run([PY, "-m", "pip", "install", "--no-deps", "hume-tada"])
    run([PY, "-m", "pip", "install", "git+https://github.com/QwenLM/Qwen3-TTS.git"])
    # Verify everything needed by our remote Voicebox path before launch.
    run([PY, "-c", "import torch, fastapi, uvicorn, fastmcp, httpx, qwen_tts; print('Core imports OK'); print('CUDA in Voicebox env:', torch.cuda.is_available())"])
    MARKER.write_text("ready\n")

print("Voicebox backend installation ready.")


In [ ]:
# Start Voicebox, protect it with a temporary token, and create a free HTTPS Quick Tunnel.
import os, subprocess, time, pathlib, secrets, re, urllib.request, html
from IPython.display import display, HTML
import requests

ROOT = pathlib.Path("/content/voicebox")
PY = "/content/voicebox-py311/bin/python"
if not pathlib.Path(PY).exists():
    raise RuntimeError("Voicebox Python environment is missing. Run the install cell above first.")

for name in ("VOICEBOX_PROCESS", "VOICEBOX_PROXY", "VOICEBOX_TUNNEL"):
    old = globals().get(name)
    if old is not None:
        try:
            old.terminate()
        except Exception:
            pass

token = secrets.token_urlsafe(32)
env = os.environ.copy()
env["VOICEBOX_CORS_ORIGINS"] = "https://epubplayer-eta.vercel.app"
env["PYTHONUNBUFFERED"] = "1"
voicebox_log = open("/content/voicebox-backend.log", "w")
VOICEBOX_PROCESS = subprocess.Popen(
    [PY, "-m", "backend.main", "--host", "127.0.0.1", "--port", "17493", "--data-dir", "/content/voicebox-data"],
    cwd=str(ROOT), env=env, stdout=voicebox_log, stderr=subprocess.STDOUT
)

for _ in range(180):
    try:
        r = requests.get("http://127.0.0.1:17493/health", timeout=2)
        if r.ok:
            print("Voicebox health:", r.json())
            break
    except Exception:
        pass
    if VOICEBOX_PROCESS.poll() is not None:
        log = pathlib.Path("/content/voicebox-backend.log").read_text(errors="replace")
        raise RuntimeError("Voicebox failed to start:\n\n" + log[-8000:])
    time.sleep(1)
else:
    log = pathlib.Path("/content/voicebox-backend.log").read_text(errors="replace")
    raise RuntimeError("Voicebox did not become healthy:\n\n" + log[-8000:])

proxy_code = r"""
import os, httpx
from fastapi import FastAPI, Request
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
TOKEN = os.environ["VOICEBOX_EPUB_TOKEN"]
UPSTREAM = "http://127.0.0.1:17493"
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["https://epubplayer-eta.vercel.app"], allow_credentials=False, allow_methods=["*"], allow_headers=["*"])

@app.api_route("/{path:path}", methods=["GET","POST","PUT","DELETE","PATCH","OPTIONS"])
async def proxy(path: str, request: Request):
    if request.method != "OPTIONS" and request.headers.get("x-voicebox-token") != TOKEN:
        return Response("Unauthorized", status_code=401)
    body = await request.body()
    headers = {k:v for k,v in request.headers.items() if k.lower() not in {"host","content-length","x-voicebox-token","origin","referer"}}
    async with httpx.AsyncClient(timeout=600.0) as client:
        resp = await client.request(request.method, f"{UPSTREAM}/{path}", params=request.query_params, content=body, headers=headers)
    out_headers = {k:v for k,v in resp.headers.items() if k.lower() not in {"content-length","content-encoding","transfer-encoding","connection","access-control-allow-origin","access-control-allow-credentials"}}
    return Response(resp.content, status_code=resp.status_code, headers=out_headers, media_type=resp.headers.get("content-type"))
"""
pathlib.Path("/content/voicebox_secure_proxy.py").write_text(proxy_code)
proxy_env = os.environ.copy()
proxy_env["VOICEBOX_EPUB_TOKEN"] = token
proxy_log = open("/content/voicebox-proxy.log", "w")
VOICEBOX_PROXY = subprocess.Popen(
    [PY, "-m", "uvicorn", "voicebox_secure_proxy:app", "--host", "127.0.0.1", "--port", "7860"],
    cwd="/content", env=proxy_env, stdout=proxy_log, stderr=subprocess.STDOUT
)

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:7860/health", headers={"X-Voicebox-Token": token}, timeout=2).ok:
            break
    except Exception:
        pass
    if VOICEBOX_PROXY.poll() is not None:
        log = pathlib.Path("/content/voicebox-proxy.log").read_text(errors="replace")
        raise RuntimeError("Secure proxy failed:\n\n" + log[-5000:])
    time.sleep(1)
else:
    log = pathlib.Path("/content/voicebox-proxy.log").read_text(errors="replace")
    raise RuntimeError("Secure proxy did not start:\n\n" + log[-5000:])

cloudflared = pathlib.Path("/content/cloudflared")
if not cloudflared.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
    cloudflared.chmod(0o755)

VOICEBOX_TUNNEL = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

tunnel_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = VOICEBOX_TUNNEL.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break
if not tunnel_url:
    raise RuntimeError("Cloudflare tunnel did not provide a URL.")

from urllib.parse import urlencode
pair = "https://epubplayer-eta.vercel.app/app/settings#" + urlencode({"voiceboxUrl": tunnel_url, "voiceboxToken": token})
print("VOICEBOX SERVER:", tunnel_url)
print("ACCESS TOKEN:", token)
display(HTML(f"""<div style="font-family:-apple-system;padding:18px;border:1px solid #ddd;border-radius:14px"><h2>Voicebox is ready</h2><p>Keep this Colab runtime running while you listen.</p><a href="{html.escape(pair)}" target="_blank" style="display:inline-block;padding:14px 18px;background:#6d5dfc;color:white;text-decoration:none;border-radius:12px;font-weight:700">Connect EPUB Player to Voicebox</a></div>"""))
